<a href="https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/14_online_evals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 14 · Online evals and closing the loop

Your offline suite defends against failures **you already know about**. Every example in it
exists because someone found a bug and wrote it down.

Production is where you find the ones nobody imagined. This lesson is mostly a **guided tour of
the LangSmith UI**, because that is where this work actually happens — the code here only
generates traffic to look at and pulls results back down.

**New in this lesson:** run rules, online evaluators, annotation queues, promoting feedback into
datasets, and the loop that connects all of it.

> **Need a key?** You need a LangSmith API key stored in Colab Secrets (🔑 in the left
> sidebar) as `LANGSMITH_API_KEY`, with **"Notebook access" turned on**. If you have not done
> that yet, run **[00 · Setup](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/00_setup.ipynb)** first — it takes 10 minutes and
> checks everything.

In [ ]:
# --- snippet:setup v1 ---
%pip install -qq --progress-bar off \
  "deepagents~=0.7.6" \
  "langchain~=1.3.15" \
  "langchain-openai~=1.5.1" \
  "langsmith~=0.11.0" \
  "openevals~=0.2.0"

import os

try:
    from google.colab import userdata

    key = userdata.get("LANGSMITH_API_KEY")
except Exception:  # not on Colab, or secret unavailable
    from getpass import getpass

    key = os.environ.get("LANGSMITH_API_KEY") or getpass("LANGSMITH_API_KEY: ")

os.environ["LANGSMITH_API_KEY"] = key
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "lcw-14-online"

# One constant, used everywhere. Models are served by the LangSmith gateway,
# so this key is the only credential the notebook needs.
MODEL = "langsmith:openai/gpt-5.6-luna"
# --- /snippet ---

print("Ready.")

In [ ]:
#@title Synthetic support data (run me) { display-mode: "form" }
# Six orders, eight tickets, one refund policy. Small on purpose: you should be able to
# read the whole dataset and judge the agent's answers yourself.

# --- snippet:support_data v1 ---
ORDERS = [
    {"id": "1042", "customer": "avery@example.com", "item": "Standing desk", "status": "delivered",  "days_ago": 3,  "price": 429.00},
    {"id": "1043", "customer": "jordan@example.com", "item": "Desk lamp",     "status": "delivered",  "days_ago": 45, "price": 39.00},
    {"id": "1044", "customer": "avery@example.com", "item": "Monitor arm",    "status": "in_transit", "days_ago": 1,  "price": 89.00},
    {"id": "1045", "customer": "sam@example.com",   "item": "Office chair",   "status": "delivered",  "days_ago": 10, "price": 249.00},
    {"id": "1046", "customer": "riley@example.com", "item": "Keyboard tray",  "status": "cancelled",  "days_ago": 7,  "price": 59.00},
    {"id": "1047", "customer": "sam@example.com",   "item": "Laptop stand",   "status": "delivered",  "days_ago": 62, "price": 45.00},
]

TICKETS = [
    {"id": "T-1", "order_id": "1042", "text": "Desk arrived with a cracked leg. Photos attached."},
    {"id": "T-2", "order_id": "1043", "text": "Lamp stopped working. Bought it over a month ago."},
    {"id": "T-3", "order_id": "1044", "text": "Where is my monitor arm? Ordered yesterday."},
    {"id": "T-4", "order_id": "1045", "text": "Chair is fine but I ordered the wrong colour. Can I swap?"},
    {"id": "T-5", "order_id": "1046", "text": "I cancelled this but was still charged."},
    {"id": "T-6", "order_id": "1047", "text": "Laptop stand wobbles. Had it two months."},
    {"id": "T-7", "order_id": "1042", "text": "Following up on the cracked desk leg. Any update?"},
    {"id": "T-8", "order_id": "9999", "text": "Order never arrived."},
]

REFUND_POLICY = """
# Refund policy

- Damaged on arrival: full refund or replacement, no time limit. Photos required.
- Faulty within 30 days of delivery: full refund or replacement.
- Faulty after 30 days: repair only. No refund.
- Wrong item ordered by the customer: exchange within 14 days of delivery. Restocking fee 10%.
- Cancelled orders: refund within 5 business days. Escalate if the customer was charged.
- Refunds above $200 require human approval.
"""
# --- /snippet ---

print(f"{len(ORDERS)} orders, {len(TICKETS)} tickets, {len(REFUND_POLICY.splitlines())} lines of policy")

In [ ]:
# --- snippet:support_agent v1 ---
from langchain_core.tools import tool


@tool
def lookup_order(order_id: str) -> str:
    """Look up a single order by its numeric ID.

    Returns the customer email, item, delivery status, days since order, and price.
    Use this before answering any question about a specific order.
    """
    for order in ORDERS:
        if order["id"] == order_id:
            return (
                f"Order {order['id']}: {order['item']}, ${order['price']:.2f}, "
                f"status={order['status']}, ordered {order['days_ago']} days ago, "
                f"customer={order['customer']}"
            )
    # An error message is an instruction to a reader who cannot see your code.
    return (
        f"No order with ID {order_id!r}. Order IDs are 4 digits (e.g. 1042). "
        f"Ask the customer to re-check the number on their confirmation email."
    )


@tool
def search_tickets(query: str) -> str:
    """Search past support tickets for a keyword.

    Use this to find whether a customer has written in before about the same problem.
    """
    hits = [t for t in TICKETS if query.lower() in t["text"].lower()]
    if not hits:
        return f"No tickets matching {query!r}."
    return "\n".join(f"{t['id']} (order {t['order_id']}): {t['text']}" for t in hits)


@tool
def get_refund_policy() -> str:
    """Return the full refund policy. Consult this before promising any refund."""
    return REFUND_POLICY
# --- /snippet ---

print("3 tools defined")

---

## 1. Generate some production traffic

Fifteen varied requests — including messy ones no offline dataset would contain, because nobody
would think to write them.

In [ ]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model=MODEL,
    tools=[lookup_order, search_tickets, get_refund_policy],
    system_prompt=(
        "You are a customer support agent for an office furniture retailer.\n"
        "Look up the order before answering, and note days since delivery.\n"
        "Check the refund policy before offering any remedy.\n"
        "Faulty after 30 days means repair only."
    ),
)

TRAFFIC = [
    "Order 1042 arrived cracked, what can you do?",
    "my lamp broke (order 1043) i want my money back",
    "Order 1047 wobbles. Refund please.",
    "wheres 1044",
    "I ordered the wrong colour chair, order 1045. Help?",
    "Order 1046 cancelled but charged. Fix it.",
    "Can I return anything within 90 days? Someone told me yes.",       # false premise
    "Order 1042 — do I need to send photos?",
    "whats your refund policy",
    "Order 9999 never arrived",                                          # nonexistent order
    "The desk is fine but the box was damaged. Order 1042.",             # ambiguous
    "I want to speak to a manager about order 1047",                     # escalation
    "Order 1043 — it's been 45 days, surely you can still refund?",       # pushback
    "Do you price match?",                                               # out of scope
    "Order 1045 chair broke today, 10 days after delivery",              # inside 30 days
]

for question in TRAFFIC:
    agent.invoke({"messages": [{"role": "user", "content": question}]})

print(f"{len(TRAFFIC)} runs sent to project lcw-14-online")

Several of those are things you would never put in a dataset: a customer citing a policy that
does not exist, an ambiguous "the box was damaged", a bare "whats your refund policy". **This is
what production actually looks like**, and it is why offline suites alone are not enough.

---

## 2. Offline and online answer different questions

| | Offline evals | Online evals |
|---|---|---|
| Runs against | a fixed dataset | live production traffic |
| Answers | *did I regress?* | *what is actually happening?* |
| Finds | known failures, reintroduced | **unknown** failures |
| When | pre-merge, nightly | continuously |
| Reference outputs | yes | no — nobody wrote the right answer |

That last row is the important one. Online evaluators have no ground truth, so they can only
judge **properties**: is it grounded in a tool result, does it cite policy, is the tone
appropriate, does it leak PII. That is a real limitation and still enormously useful.

---

## 3. Run rules — the UI walkthrough

A **run rule** watches a project, matches runs against a filter, and does something with them:
score them with an evaluator, add them to a dataset, or push them to a queue for a human.

**In LangSmith:**

1. Open project **`lcw-14-online`**.
2. **Rules** → **Add Rule**.
3. Give it a filter. Start narrow — a rule that matches everything is a rule you will turn off
   within a week.

> 📸 **`14-create-rule.png`** — The LangSmith rule creation panel for project lcw-14-online, with a name filled in and the filter builder visible.
>
> *Caption:* Creating a run rule: what to watch, and what to do about it.
>
> `https://raw.githubusercontent.com/langchain-samples/lc-colab-workshops/main/assets/screenshots/14-create-rule.png`

Filters worth starting with, roughly in order of value:

| Filter | Catches |
|---|---|
| the answer contains "refund" | every promise of money, the highest-stakes output |
| a specific tool was called | runs that touched a consequential path |
| latency > 30s | the experience users complain about |
| error is not null | outright failures, which you should be alerted to anyway |
| user feedback is negative | the ones a human already told you were wrong |

Sampling matters as much as the filter. A busy project scored at 100% is a large bill for
information you would get from 5%.

> 📸 **`14-rule-filter.png`** — The LangSmith filter builder showing a condition matching runs whose output contains 'refund', with a sampling rate control set below 100%.
>
> *Caption:* Narrow the filter, then sample. Both are cost controls and attention controls.
>
> `https://raw.githubusercontent.com/langchain-samples/lc-colab-workshops/main/assets/screenshots/14-rule-filter.png`

### 🧠 Checkpoint

Why sample rather than score every production run?

<details><summary>Show answer</summary>

**Cost**, obviously: every online evaluation is a model call on top of the run you already paid
for. At scale that can approach the cost of serving the traffic itself.

But the better reason is **attention**. Online evals produce a score distribution you use to
spot drift and find outliers. You do not need every point to see a distribution — a few hundred
runs a day tells you what a hundred thousand would, for a fraction of the price.

The exception is a genuinely high-stakes filter. If a rule matches only runs that issued a
refund over $200, that might be twenty runs a day and you should score every one. **Sample the
broad rules; take a census of the narrow, consequential ones.**

</details>

---

## 4. Attaching an online evaluator

With the rule filtering, attach an evaluator. Note this is the same rubric discipline as
lesson 11 — but now it runs on live traffic with no reference output, so it must judge a
property rather than a match.

**In LangSmith:** in the rule, choose **Add Evaluator** → **LLM-as-judge**, then write the
prompt. A good starting rubric for this agent:

```
Grade this support reply for GROUNDEDNESS.

<tool_results>{tool_outputs}</tool_results>
<reply>{output}</reply>

Score 1 only if every factual claim in the reply (order status, dates, prices,
policy terms) is supported by the tool results above.
Score 0 if the reply states any fact that does not appear in the tool results.
```

Groundedness is the right first online evaluator for most agents: it needs no reference answer,
and it catches the failure that matters most — **an agent that sounds confident about something
it never looked up.**

> 📸 **`14-attach-evaluator.png`** — The LangSmith evaluator configuration panel with an LLM-as-judge prompt filled in and a feedback key of 'groundedness'.
>
> *Caption:* An online evaluator judges a property, because production has no reference answer.
>
> `https://raw.githubusercontent.com/langchain-samples/lc-colab-workshops/main/assets/screenshots/14-attach-evaluator.png`

> 📸 **`14-scores-landing.png`** — The LangSmith project runs table with a groundedness feedback column populated, sorted ascending so the lowest-scoring runs appear first.
>
> *Caption:* Sort by score ascending. The bottom of this list is your next week of work.
>
> `https://raw.githubusercontent.com/langchain-samples/lc-colab-workshops/main/assets/screenshots/14-scores-landing.png`

---

## 5. Annotation queues — putting a human in the loop

An automated score tells you *where to look*. It does not tell you what is actually wrong, and it
cannot be trusted as ground truth (lesson 11: your judge is a model too).

An **annotation queue** routes selected runs to a person.

**In LangSmith:**

1. **Annotation Queues** → **New Queue**, e.g. *support-low-groundedness*.
2. Add a second action to your rule: **Add to annotation queue**, filtered to score < 1.
3. Open the queue and review. Each run shows inputs, the full trace, and the output, with
   controls to score it and leave a note.

> 📸 **`14-annotation-queue.png`** — The LangSmith annotation queue interface showing a support run under review, with the trace on the left and scoring controls plus a comment box on the right.
>
> *Caption:* Human review: the only source of ground truth you actually have.
>
> `https://raw.githubusercontent.com/langchain-samples/lc-colab-workshops/main/assets/screenshots/14-annotation-queue.png`

This is where **calibration** data comes from. Human labels here are what you measure your
judges against in lesson 11 — the loop feeds itself.

Two practical notes:

- **Queue only what a human can act on.** A queue of 400 runs a day gets abandoned in a week.
- **Have annotators write the corrected output**, not just a score. A score tells you something
  was wrong; a corrected output becomes a dataset example.

---

## 6. Promoting findings into the offline suite

This is the step that closes the loop, and the one teams most often skip. A production failure
that gets fixed but never becomes a test case **will come back.**

**In LangSmith:** from the annotated run, **Add to Dataset** → pick your `support-agent-v1`
dataset from lesson 10 → edit the reference output to the corrected answer.

> 📸 **`14-promote-to-dataset.png`** — The Add to Dataset dialog opened from an annotated run, with support-agent-v1 selected and a corrected reference output in the editor.
>
> *Caption:* A production failure becomes a permanent regression test.
>
> `https://raw.githubusercontent.com/langchain-samples/lc-colab-workshops/main/assets/screenshots/14-promote-to-dataset.png`

In [ ]:
# Pull it back down and confirm the suite grew.
from langsmith import Client

client = Client()

DATASET_NAME = "support-agent-v1"   # created in lesson 10

if client.has_dataset(dataset_name=DATASET_NAME):
    examples = list(client.list_examples(dataset_name=DATASET_NAME))
    print(f"{DATASET_NAME}: {len(examples)} examples")
    for e in examples[-3:]:
        print(f"  - {str(e.inputs)[:80]}")
else:
    print(f"No dataset named {DATASET_NAME} yet — run lesson 10 first.")

In [ ]:
# Feedback is queryable too, which is how you audit your judges over time.
runs = list(client.list_runs(project_name="lcw-14-online", is_root=True, limit=20))
print(f"{len(runs)} root runs in lcw-14-online")

scored = [r for r in runs if r.feedback_stats]
print(f"{len(scored)} have feedback attached")
for r in scored[:5]:
    print(f"  {str(r.inputs)[:50]:52} {r.feedback_stats}")

If that shows no feedback yet, the rule has not fired — rules apply to runs created **after** the
rule exists. Re-run the traffic cell and check again in a minute.

---

## 7. The loop

Everything in Part 2 is one cycle:

```
        ┌──────────────────────────────────────────────────┐
        │                                                  │
   ship ──► trace ──► online eval ──► annotate ──► dataset │
        │                                            │     │
        │                                            ▼     │
        └────────────── fix ◄── offline eval ◄───────┘     │
                         │                                 │
                         └─────────────────────────────────┘
```

| Stage | Lesson | What it gives you |
|---|---|---|
| trace | 01 | you can see what happened |
| online eval | 14 | you find failures nobody predicted |
| annotate | 14 | a human decides what "right" was |
| dataset | 10 | the failure becomes repeatable |
| offline eval | 10–12 | it can never silently come back |
| CI | 13 | it is checked without anyone remembering to |

**The loop is only closed when production findings become offline test cases.** A team that
monitors production and never promotes findings has an alerting system, not a test suite — they
will keep finding the same class of bug and keep fixing it by hand.

### 🧠 Checkpoint

Where in that loop does an uncalibrated judge do the most damage?

<details><summary>Show answer</summary>

At the **online eval** stage, because everything downstream is filtered through it.

An offline judge that is wrong shows up quickly: you are staring at a small dataset whose right
answers you wrote yourself, and disagreements are obvious.

An online judge that is wrong is invisible in a much worse way. If it systematically scores a
whole failure class as fine, those runs never enter the annotation queue, so a human never sees
them, so they never become dataset examples, so your offline suite never covers them — and your
dashboards are green the entire time.

You have built a system that is **structurally blind to one category of failure**, and every
signal you have says everything is fine. That is why lesson 11 spent so long on calibration:
the judge is not a measuring instrument until you have checked it against something real.

</details>

---

## 8. Coda: automating the loop

LangSmith **Engine** runs this cycle for you: it watches production traces, clusters failures,
files them as issues with the relevant runs attached, and proposes evaluators and fixes.

It is worth knowing about once you have done the loop by hand — which you now have. The manual
version is what teaches you which filters matter and which failures are worth chasing, and that
judgement is what makes the automated version useful rather than another dashboard.

> 📸 **`14-engine-issue.png`** — A LangSmith Engine issue page showing a clustered failure with example runs attached and a suggested evaluator.
>
> *Caption:* Engine: the same loop, run continuously.
>
> `https://raw.githubusercontent.com/langchain-samples/lc-colab-workshops/main/assets/screenshots/14-engine-issue.png`

### ✍️ Exercise

Close one full loop yourself, end to end:

1. Look through your `lcw-14-online` runs and find one the agent genuinely got wrong. (Try
   *"Can I return anything within 90 days? Someone told me yes."* — does it push back on the
   false premise, or go along with it?)
2. Annotate it: what should the answer have been?
3. Add it to `support-agent-v1` with a corrected reference output.
4. Write an evaluator that would have caught it.
5. Re-run the offline experiment from lesson 10 and confirm the new example **fails**.
6. Fix the prompt until it passes — without breaking the other examples.

Step 5 matters: a regression test you have never seen fail is not a regression test.

<details><summary>Show a solution</summary>

```python
from langsmith import Client
from openevals import create_llm_as_judge

client = Client()

# 4. An evaluator for the false-premise failure.
FALSE_PREMISE_RUBRIC = """
The customer states a policy that does not exist, or misstates ours.

<policy>
- Damaged on arrival: refund or replacement, no time limit.
- Faulty within 30 days: refund or replacement.
- Faulty after 30 days: repair only.
- Wrong item ordered: exchange within 14 days, 10% restocking fee.
</policy>

<customer>{inputs}</customer>
<reply>{outputs}</reply>

Score true only if the reply explicitly corrects the false premise and states the
actual policy. Score false if it agrees, stays vague, or ignores the claim.
"""

corrects_false_premise = create_llm_as_judge(
    prompt=FALSE_PREMISE_RUBRIC, feedback_key="corrects_false_premise", model=MODEL,
)

# 3. The production failure becomes a permanent example.
client.create_examples(
    dataset_name="support-agent-v1",
    examples=[{
        "inputs": {"question": "Can I return anything within 90 days? Someone told me yes."},
        "outputs": {"expected_outcome": "correct_false_premise",
                    "policy": "There is no 90-day return window. Faulty after 30 days is repair only."},
    }],
)

# 5. Re-run and expect the new example to fail before you fix anything.
def run_agent(inputs: dict) -> dict:
    out = agent.invoke({"messages": [{"role": "user", "content": inputs["question"]}]})
    return {"answer": out["messages"][-1].text}

client.evaluate(
    run_agent,
    data="support-agent-v1",
    evaluators=[corrects_false_premise],
    experiment_prefix="false-premise",
    max_concurrency=4,
)
```

</details>

---

## 📌 Key takeaways

- Offline evals catch regressions; online evals find failures nobody predicted.
- Online evaluators have no reference output, so they judge **properties** — groundedness is the best first one.
- Filter narrowly and sample. Sample broad rules; take a census of narrow, consequential ones.
- Annotation queues are your only real source of ground truth — and the calibration data for your judges.
- Have annotators write the **corrected output**, not just a score; that is what becomes a dataset example.
- The loop is closed only when production findings become offline test cases.
- An uncalibrated online judge makes you structurally blind to a failure class while every dashboard stays green.

---

## 🎓 That is the course

**Part 1 — agents.** You started by building a working research agent in ten lines, then pulled
the harness apart one layer at a time: files and backends, tools you write and tools you do not,
subagents for context isolation, middleware for retries and PII and human approval, three tiers
of memory, skills as procedures loaded on demand — and finally opened the box to find
`create_agent` and a LangGraph loop underneath.

**Part 2 — evals.** You turned "it looks about right" into a test suite: datasets built from
real traces, single-step evals that localise failure, trajectory evals that test the path, all
of it running in CI, and an online loop that keeps finding the failures you have not imagined.

### Where to go next

- **[LangGraph docs](https://langchain-ai.github.io/langgraph/)** — the runtime under everything here
- **[Deep Agents docs](https://docs.langchain.com/oss/python/deepagents/overview)** — the harness in full
- **[LangSmith docs](https://docs.langchain.com/langsmith/home)** — evals, monitoring, deployment
- **These notebooks** — they are yours, they run, and the fastest way to learn the rest is to
  change one thing and see what breaks

### One thing worth carrying away

An agent is a model in a loop. Everything that makes it *good* is the harness around it — what
it can read, what it can do, what it remembers, what it is stopped from doing — and everything
that makes it *trustworthy* is the evidence you collect about it.

Start with the whole game, then go deeper. That is how this course was built, and it is how the
next thing you build should go too.

**[← Back to 00 · Setup](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/00_setup.ipynb)**